# BUS-BRA blank-image controls: **5 white + 5 black per model**

**Purpose:** Measure the response stability and color dependence of **Qwen2-VL-2B-Instruct** and **MedVLM-R1** when no diagnostic visual content is present.

- Uses exactly the **same prompt**, model checkpoints, BF16 path, and greedy decoding as the completed real-image experiment.
- Performs **5 identical solid-white** and **5 identical solid-black** 512 × 512 RGB image inferences **per model** (20 VLM inferences total across 2 models).
- Preserves all 400 previously frozen BUS-BRA sample IDs/labels to construct **derived 400-row control CSVs** if and only if all five repetitions for that model/color agree on both raw output and parsed prediction.
- Produces two independent result folders (`white/` and `black/`) matching the existing real-vs-blank paired analysis notebook's input schema.
- **Does not infer on any original ultrasound image and does not re-run the 400 real-image experiment.** Original image files are only accessed for the original frozen sample/manifest verification.

**Interpretation:** Repeated constant inputs are **stability checks**, not independent labeled test cases. A 400-row derived CSV provides a matched **constant-prediction baseline**; it must not be represented as 400 independent control inferences.


## 1. Configuration and logging

Attach the same BUS-BRA dataset as the original Kaggle run. The control images are synthesized in RAM; the original ultrasound image **never enters the VLM**. Set `DATA_ROOT` only if dataset autodiscovery is ambiguous.


In [ ]:
from pathlib import Path

# Two solid-color controls; each is repeated five times per model for stability.
JOBS = [(key, "bf16") for key in ('qwen2_vl', 'medvlm_r1')]
BLANK_IMAGE_SIZE = (512, 512)   # (width, height), constant across all samples.
CONTROL_COLORS = {"white": (255, 255, 255), "black": (0, 0, 0)}
TRIALS_PER_COLOR = 5
assert BLANK_IMAGE_SIZE == (512, 512) and TRIALS_PER_COLOR >= 2
DATA_ROOT = None          # Auto-discover bus_data.csv under /kaggle/input.
INPUT_ROOT = Path("/kaggle/input")
# Control outputs are separate from the original-image experiment (no overwriting).
CSV_ROOT = Path("/kaggle/working/blank_image_white_black_controls")
OUTPUT_ROOT = Path("/tmp/bus_bra_dual_blank_control_qwen_medvlm_state")  # Temporary state/resume.
EXPORT_TAGS = {"qwen2_vl": "qwen2vl_2b", "medvlm_r1": "medvlm_r1",
               "gemma3": "gemma3_4b", "medgemma": "medgemma_4b"}
EXTRACT_ROOT = Path("/tmp/bus_bra_extracted")
MODEL_CACHE_ROOT = Path("/tmp/bus_bra_model_cache")
CLEAR_MODEL_CACHE_AFTER_LAST_JOB = True  # Remove this notebook's downloaded weights to save disk.
INSTALL_DEPENDENCIES = True

# Each execution has its own logs, so restored results cannot overwrite active logs.
import codecs
import os
import re
import selectors
import signal
import subprocess
import sys
import threading
import time
import uuid
from contextlib import contextmanager
from datetime import datetime, timezone

print(f"[{datetime.now(timezone.utc).isoformat()}] Notebook execution started", flush=True)
LOG_SESSION_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
LOG_ROOT = OUTPUT_ROOT / "logs" / LOG_SESSION_ID
LOG_ROOT.mkdir(parents=True, exist_ok=True)
PROGRESS_LOG = LOG_ROOT / "progress.log"
INSTALL_LOG = LOG_ROOT / "install.log"
INSTALL_LOG.touch(exist_ok=True)
INSTALL_TIMEOUT_SECONDS = 15 * 60
HEARTBEAT_SECONDS = 30
_LOG_LOCK = threading.Lock()

def redact_log(value):
    text = str(value)
    token = globals().get("HF_TOKEN")
    if token:
        text = text.replace(token, "[REDACTED]")
    text = re.sub(r"hf_[A-Za-z0-9]+", "[REDACTED]", text)
    return re.sub(r"(https?://)[^\s/@]+:[^\s/@]+@", r"\1[REDACTED]@", text)

def log(*parts):
    message = redact_log(" ".join(str(p) for p in parts))
    line = f"[{datetime.now(timezone.utc).isoformat(timespec='seconds')}] {message}"
    with _LOG_LOCK:
        with PROGRESS_LOG.open("a", encoding="utf-8") as stream:
            stream.write(line + "\n")
        print(line, flush=True)

@contextmanager
def log_stage(name):
    """A heartbeat means the call is still waiting, not that it is making progress."""
    started = time.monotonic()
    stop = threading.Event()
    log("START", name)
    def heartbeat():
        while not stop.wait(HEARTBEAT_SECONDS):
            log("WAIT", name, f"({time.monotonic() - started:.0f}s elapsed; call has not returned)")
    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    try:
        yield
    except BaseException as exc:
        log("FAILED", name, type(exc).__name__, str(exc))
        raise
    else:
        log("DONE", name, f"({time.monotonic() - started:.1f}s)")
    finally:
        stop.set()
        worker.join(timeout=2)

def run_logged_install(command, timeout_seconds=INSTALL_TIMEOUT_SECONDS):
    """Stream the child process into Jupyter stdout; bound the entire installation."""
    started = time.monotonic()
    # A separate process group lets timeout/interrupt cleanup stop build children too.
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               start_new_session=True, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    pending = ""
    eof = False
    def emit(line):
        clean = redact_log(line)
        with INSTALL_LOG.open("a", encoding="utf-8") as stream:
            stream.write(clean + "\n")
        log("pip |", clean)
    try:
        with selectors.DefaultSelector() as selector:
            selector.register(process.stdout, selectors.EVENT_READ)
            while not eof or process.poll() is None:
                if time.monotonic() - started >= timeout_seconds:
                    raise TimeoutError(f"Dependency installation exceeded {timeout_seconds}s. See {INSTALL_LOG}")
                for key, _ in selector.select(timeout=0.2):
                    chunk = os.read(key.fileobj.fileno(), 65536)
                    if not chunk:
                        pending += decoder.decode(b"", final=True)
                        if pending:
                            emit(pending)
                            pending = ""
                        selector.unregister(key.fileobj)
                        eof = True
                        continue
                    pending += decoder.decode(chunk).replace("\r", "\n")
                    while "\n" in pending:
                        line, pending = pending.split("\n", 1)
                        if line:
                            emit(line)
            if process.returncode != 0:
                raise RuntimeError(f"Dependency installation failed (exit {process.returncode}). See {INSTALL_LOG}")
    finally:
        # Also runs on notebook interruption, timeout, or output-processing failure.
        try:
            os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        process.wait(timeout=5)
        process.stdout.close()

log("Notebook execution started; progress log:", PROGRESS_LOG)
log("Python:", sys.version.split()[0], "| dependency timeout:", INSTALL_TIMEOUT_SECONDS, "seconds")
log("BLANK IMAGE CONTROLS: 5 white + 5 black per model, total 20 inferences; 400 derived rows per stable condition")

SEED = 42
FINAL_SIZE = 400
MAX_NEW_TOKENS = 32       # Agreed initial limit; never auto-increased.
GPU_MEMORY_FRACTION = 0.80  # Reserve space for image activations / KV cache.
CODE_VERSION = "bus-bra-white-black-stability-v2-bf16-greedy"
PARSER_VERSION = "strict-final-v2-terminal-eos"
log("Final code version:", CODE_VERSION, "| parser:", PARSER_VERSION)

# Both archives were compared locally: all 1,875 images and all ten shared
# metadata fields match exactly. Extra Kaggle columns are not used for inference.
VERIFIED_DATASET_VARIANTS = {
    "original": {
        "metadata_sha256": "f218d0f10b0aeeb14ff4b2cfed8631b2f4a7cf608e564548b35bddd55f3aafb3",
        "source_fingerprint": "e1fadc2a04595e1e17c9e8e7e72def7ffa954d52b4de5c386398623286848b49",
        "unused_extra_columns": [],
    },
    "kaggle_orvile": {
        "metadata_sha256": "1a7756e34fa27cda4faa48f41c1b5a5443e2bd0af8aa2a41e24ad8b1a96ede71",
        "source_fingerprint": "180c04afcd37b5fb6a93900ec48bc2dec9234d8e5216e39ba1a075d9e6d50f1d",
        "unused_extra_columns": ["HOB", "HOP", "K5B", "K5P", "K10B", "K10P"],
    },
}

MODEL_IDS = {
    "qwen2_vl": "Qwen/Qwen2-VL-2B-Instruct",
    "medvlm_r1": "JZPeterPan/MedVLM-R1",
    "gemma3": "google/gemma-3-4b-it",
    "medgemma": "google/medgemma-4b-it",
}
# Frozen to the completed v4 smoke test; do not resolve a moving main branch.
MODEL_REVISIONS = {
    "gemma3": "093f9f388b31de276ce2de164bdc2081324b9767",
    "medgemma": "290cda5eeccbee130f987c4ad74a59ae6f196408",
    "medvlm_r1": "d256f2cfdf98c6872c1dc9f20b7dd52f49374fe9",
    "qwen2_vl": "895c3a49bc3fa70a340399125c650a463535e71c"
}
PACKAGE_PINS = {
    "transformers": "4.57.1", "accelerate": "1.11.0",
    "bitsandbytes": "0.48.1", "huggingface-hub": "0.36.0",
    "tokenizers": "0.22.1", "safetensors": "0.6.2",
    "sentencepiece": "0.2.1",
}
# Preserve Kaggle's torch/torchvision and Pillow; resolve and record them in setup.
BUS_BRA_PROMPT = """You are a medical assistant. Examine this breast ultrasound image.
Classify the lesion as exactly ONE of the following:
- benign: non-cancerous lesion
- malignant: cancerous lesion

Respond with ONLY the label: benign or malignant."""

assert FINAL_SIZE == 400 and MAX_NEW_TOKENS == 32
assert JOBS and len(JOBS) == len(set(JOBS))
for key, precision in JOBS:
    assert key in MODEL_IDS
    assert precision in {"bf16"}, "Final inference is BF16 only; no automatic precision fallback"


# SHA256 of code cells joined by two newlines, with this digest normalized to 64 zeros.
IMPLEMENTATION_SHA256 = "57948fe3acb98101a170eb7d895d57abe786c6ace99216b4e2cbd6fcfb1b0bbd"
CODE_GIT_COMMIT = None  # This derived control notebook has no separately verified Git commit.


## 2. Kaggle dependencies

Keep the pinned inference stack from the original notebook. Kaggle's `torch`, `torchvision`, and Pillow are preserved.


In [ ]:
with log_stage("Dependency installation and version verification"):
    import importlib.metadata as im
    import subprocess
    import sys
    import PIL

    PRESERVED_RUNTIME_VERSIONS = {
        name: im.version(name) for name in ("torch", "torchvision", "Pillow")
    }
    def verify_runtime_pillow():
        expected = PRESERVED_RUNTIME_VERSIONS["Pillow"]
        installed = im.version("Pillow")
        loaded = getattr(PIL, "__version__", None)
        if installed != expected or loaded != expected:
            raise RuntimeError(
                f"Pillow runtime mismatch: preserved={expected}, installed={installed}, "
                f"loaded={loaded}. Start a fresh Kaggle run; do not reload Pillow in place."
            )
    verify_runtime_pillow()
    log("Preserving Kaggle runtime packages:", PRESERVED_RUNTIME_VERSIONS)

    if INSTALL_DEPENDENCIES:
        # Only the explicitly pinned packages may need replacement; Pillow stays intact.
        changed_loaded = []
        for package, module in {"transformers": "transformers",
                                "tokenizers": "tokenizers"}.items():
            try:
                if module in sys.modules and im.version(package) != PACKAGE_PINS[package]:
                    changed_loaded.append(package)
            except im.PackageNotFoundError:
                pass
        if changed_loaded:
            raise RuntimeError("Restart the Kaggle session before installing: " + ", ".join(changed_loaded))
        # Protect the CUDA runtime and preloaded Pillow from pip replacement.
        constraints = Path("/tmp/bus_bra_runtime_constraints.txt")
        constraints.write_text("\n".join(
            f"{name}=={version}" for name, version in PRESERVED_RUNTIME_VERSIONS.items()
        ) + "\n")
        log("Installing pinned dependencies; pip output:", INSTALL_LOG)
        run_logged_install([sys.executable, "-u", "-m", "pip", "install",
                            "--disable-pip-version-check", "--progress-bar", "off",
                            "--no-input", "--timeout", "60", "--retries", "2", "-c", str(constraints),
                            *[f"{name}=={version}" for name, version in PACKAGE_PINS.items()]])
        importlib = __import__("importlib")
        importlib.invalidate_caches()
    for name, expected in {**PACKAGE_PINS, **PRESERVED_RUNTIME_VERSIONS}.items():
        if im.version(name) != expected:
            raise RuntimeError(f"Dependency mismatch: {name}; expected {expected}.")
    verify_runtime_pillow()
    log("Inference dependencies verified. Kaggle torch/torchvision/Pillow were preserved.")

## 3. Verify the frozen 400-case manifest

The experiment still checks the exact original sample IDs, labels, and image hashes for pairing against your completed real-image run. **No original ultrasound is provided to either VLM**. This section is metadata validation, not image inference.


In [ ]:
import csv
import gc
import hashlib
import io
import json
import os
import platform
import random
import re
import shutil
import statistics
import time
import zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from PIL import Image

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def sha_bytes(value):
    return hashlib.sha256(value).hexdigest()

def sha_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def object_hash(value):
    return sha_bytes(json.dumps(value, sort_keys=True, separators=(",", ":"),
                               ensure_ascii=False).encode())

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + "\n")
    os.replace(temp, path)

def write_csv(path, rows, fields=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fields = fields or (list(rows[0]) if rows else ["status"])
    temp = path.with_suffix(path.suffix + ".tmp")
    with temp.open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields, extrasaction="raise")
        writer.writeheader()
        writer.writerows(rows)
    os.replace(temp, path)

def read_csv(path):
    with Path(path).open(newline="", encoding="utf-8-sig") as stream:
        return list(csv.DictReader(stream))

def append_jsonl(path, row):
    with Path(path).open("a") as stream:
        stream.write(json.dumps(row, sort_keys=True, allow_nan=False) + "\n")
        stream.flush()
        os.fsync(stream.fileno())

def read_jsonl(path, repair_tail=False):
    path = Path(path)
    if not path.exists():
        return []
    lines = path.read_bytes().splitlines(keepends=True)
    rows, offset = [], 0
    for i, line in enumerate(lines):
        try:
            row = json.loads(line)
        except (ValueError, UnicodeDecodeError):
            if not (repair_tail and i == len(lines) - 1 and not line.endswith(b"\n")):
                raise ValueError(f"Corrupt complete record in {path.name}, line {i + 1}")
            # Preserve the broken tail for inspection; retain all complete records.
            path.with_suffix(".partial_tail").write_bytes(line)
            with path.open("r+b") as stream:
                stream.truncate(offset)
            break
        if not isinstance(row, dict):
            raise ValueError("JSONL records must be objects")
        rows.append(row)
        offset += len(line)
    if repair_tail and rows and path.stat().st_size and not path.read_bytes().endswith(b"\n"):
        with path.open("ab") as stream:
            stream.write(b"\n")
    return rows

def discover_dataset(data_root, input_root, extract_root):
    if data_root is not None:
        root = Path(data_root)
        candidates = list(root.rglob("bus_data.csv"))
    else:
        candidates = list(Path(input_root).rglob("bus_data.csv"))
    if len(candidates) == 1:
        return candidates[0].parent
    if len(candidates) > 1:
        raise ValueError("Multiple BUS-BRA metadata files found; set DATA_ROOT explicitly")
    if data_root is not None:
        raise FileNotFoundError("DATA_ROOT must contain the original bus_data.csv")
    archives = []
    for path in Path(input_root).rglob("*.zip"):
        if "bus" not in path.name.lower():
            continue
        with zipfile.ZipFile(path) as archive:
            if any(Path(n).name == "bus_data.csv" for n in archive.namelist()):
                archives.append(path)
    if len(archives) != 1:
        raise FileNotFoundError("Attach a BUS-BRA dataset with bus_data.csv and original Images, or its ZIP. Set DATA_ROOT if ambiguous.")
    extract_root = Path(extract_root)
    extract_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        for member in archive.infolist():
            parts = Path(member.filename).parts
            # Only original images, metadata and license; no masks or CV files needed.
            wanted = "images" in [p.lower() for p in parts] or Path(member.filename).name in {"bus_data.csv", "LICENSE.txt"}
            if member.is_dir() or not wanted:
                continue
            target = (extract_root / member.filename).resolve()
            if not target.is_relative_to(extract_root.resolve()):
                raise ValueError("Unsafe ZIP member")
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as src, target.open("wb") as dst:
                shutil.copyfileobj(src, dst)
    candidates = list(extract_root.rglob("bus_data.csv"))
    if len(candidates) != 1:
        raise ValueError("Ambiguous extracted dataset; set DATA_ROOT explicitly")
    return candidates[0].parent


MANIFEST_FIELDS = ['dataset', 'sample_id', 'image_id', 'image_path', 'group_id', 'split', 'task_type', 'question', 'options_json', 'reference_answer', 'category', 'image_sha256']

# SHA256 of RGB pixel bytes, not file encodings; records the control pixels exactly.
BLANK_IMAGE_SPECS = {
    color: {
        "mode": f"constant_solid_{color}_rgb",
        "size_px": list(BLANK_IMAGE_SIZE),
        "color_rgb": list(rgb),
        "pixel_sha256": sha_bytes(Image.new("RGB", BLANK_IMAGE_SIZE, rgb).tobytes()),
    }
    for color, rgb in CONTROL_COLORS.items()
}


In [ ]:
FROZEN_IMAGE_IDS = """
bus_0001-l bus_0003-r bus_0007-l bus_0008-r bus_0009-r bus_0010-r bus_0011-l bus_0012-r
bus_0017-s bus_0029-s bus_0030-s bus_0034-r bus_0037-s bus_0041-l bus_0042-r bus_0043-s
bus_0045-r bus_0048-l bus_0049-r bus_0050-l bus_0052-r bus_0053-l bus_0054-s bus_0057-l
bus_0058-r bus_0059-l bus_0076-r bus_0082-s bus_0084-l bus_0088-s bus_0092-r bus_0095-s
bus_0098-l bus_0099-r bus_0100-l bus_0104-s bus_0105-r bus_0106-r bus_0109-s bus_0114-s
bus_0118-l bus_0120-l bus_0122-s bus_0124-r bus_0130-r bus_0133-l bus_0135-r bus_0136-l
bus_0141-r bus_0143-r bus_0145-r bus_0146-r bus_0147-r bus_0148-l bus_0149-r bus_0156-r
bus_0157-l bus_0163-r bus_0165-l bus_0166-r bus_0171-l bus_0172-r bus_0174-l bus_0179-l
bus_0185-l bus_0186-l bus_0187-l bus_0191-l bus_0193-r bus_0197-r bus_0199-l bus_0200-s
bus_0205-l bus_0206-l bus_0208-r bus_0212-r bus_0214-l bus_0217-r bus_0218-l bus_0219-l
bus_0225-r bus_0230-r bus_0231-r bus_0232-r bus_0233-s bus_0234-l bus_0236-l bus_0239-l
bus_0240-r bus_0242-r bus_0243-r bus_0247-r bus_0249-r bus_0250-r bus_0260-l bus_0262-l
bus_0263-l bus_0264-r bus_0265-s bus_0268-r bus_0269-r bus_0274-r bus_0277-l bus_0281-l
bus_0282-r bus_0284-r bus_0286-r bus_0287-l bus_0288-r bus_0289-l bus_0290-l bus_0291-r
bus_0293-s bus_0301-r bus_0303-r bus_0310-l bus_0311-r bus_0319-l bus_0320-l bus_0321-s
bus_0324-s bus_0326-r bus_0328-r bus_0329-l bus_0331-s bus_0333-s bus_0334-l bus_0335-l
bus_0336-s bus_0340-l bus_0342-r bus_0344-r bus_0345-l bus_0351-l bus_0353-l bus_0360-l
bus_0364-r bus_0366-l bus_0368-l bus_0370-r bus_0371-l bus_0373-r bus_0374-l bus_0375-r
bus_0376-l bus_0379-l bus_0381-l bus_0384-r bus_0385-l bus_0387-r bus_0389-r bus_0390-r
bus_0392-r bus_0394-r bus_0395-l bus_0396-l bus_0397-l bus_0398-r bus_0399-r bus_0400-r
bus_0405-l bus_0410-l bus_0411-l bus_0413-l bus_0414-r bus_0415-l bus_0416-l bus_0418-r
bus_0421-r bus_0422-r bus_0426-l bus_0427-l bus_0431-r bus_0432-l bus_0438-r bus_0440-r
bus_0442-r bus_0445-r bus_0446-r bus_0448-l bus_0449-r bus_0453-l bus_0454-r bus_0455-l
bus_0456-l bus_0457-l bus_0458-l bus_0464-r bus_0465-l bus_0466-l bus_0471-l bus_0472-l
bus_0475-l bus_0484-r bus_0486-r bus_0488-l bus_0493-r bus_0497-l bus_0502-r bus_0507-l
bus_0508-l bus_0509-r bus_0513-r bus_0518-l bus_0521-l bus_0522-l bus_0531-r bus_0532-r
bus_0543-l bus_0544-l bus_0546-l bus_0549-l bus_0550-l bus_0551-r bus_0555-l bus_0560-r
bus_0566-l bus_0568-r bus_0569-l bus_0571-l bus_0574-r bus_0576-r bus_0579-l bus_0581-r
bus_0582-l bus_0590-l bus_0594-r bus_0595-r bus_0596-l bus_0600-r bus_0601-r bus_0602-l
bus_0603-r bus_0604-l bus_0605-l bus_0609-l bus_0612-r bus_0620-l bus_0626-r bus_0628-l
bus_0630-l bus_0631-l bus_0633-l bus_0639-r bus_0641-l bus_0644-r bus_0645-r bus_0647-l
bus_0657-r bus_0659-r bus_0661-r bus_0662-l bus_0663-r bus_0666-r bus_0667-r bus_0669-l
bus_0672-l bus_0677-s bus_0683-s bus_0685-l bus_0687-s bus_0697-r bus_0700-s bus_0705-s
bus_0706-s bus_0707-s bus_0709-s bus_0711-s bus_0712-s bus_0713-r bus_0721-s bus_0723-s
bus_0726-s bus_0728-s bus_0730-s bus_0731-s bus_0742-s bus_0745-s bus_0747-s bus_0748-s
bus_0752-s bus_0758-s bus_0760-s bus_0765-s bus_0766-s bus_0767-s bus_0768-s bus_0770-s
bus_0772-s bus_0791-s bus_0793-s bus_0797-s bus_0801-s bus_0803-s bus_0807-s bus_0809-s
bus_0811-s bus_0812-s bus_0816-s bus_0818-s bus_0820-s bus_0825-s bus_0826-s bus_0829-s
bus_0831-s bus_0836-s bus_0838-s bus_0841-s bus_0842-s bus_0846-s bus_0850-l bus_0854-s
bus_0858-l bus_0859-r bus_0860-l bus_0861-l bus_0862-r bus_0866-r bus_0867-r bus_0868-r
bus_0869-r bus_0874-l bus_0879-r bus_0881-l bus_0882-r bus_0886-s bus_0893-l bus_0896-r
bus_0899-s bus_0906-l bus_0911-l bus_0915-l bus_0917-s bus_0918-l bus_0927-l bus_0928-r
bus_0933-l bus_0934-l bus_0935-r bus_0937-l bus_0938-l bus_0941-r bus_0943-s bus_0944-r
bus_0945-r bus_0947-l bus_0948-r bus_0951-r bus_0952-l bus_0954-l bus_0957-s bus_0960-l
bus_0962-l bus_0964-r bus_0966-l bus_0971-r bus_0972-s bus_0973-l bus_0977-l bus_0979-r
bus_0980-r bus_0981-l bus_0982-l bus_0983-l bus_0985-l bus_0986-r bus_0988-s bus_0989-l
bus_0991-s bus_0993-r bus_0994-r bus_0997-l bus_0998-s bus_1002-l bus_1003-r bus_1007-l
bus_1008-r bus_1012-l bus_1013-r bus_1014-l bus_1016-l bus_1017-l bus_1023-r bus_1024-r
bus_1027-l bus_1028-r bus_1032-l bus_1033-l bus_1035-l bus_1037-r bus_1038-l bus_1042-l
bus_1046-l bus_1052-l bus_1055-l bus_1056-r bus_1058-l bus_1060-s bus_1061-r bus_1063-r
""".split()
PILOT_CASES = ['1004', '1049', '184', '196', '228', '253', '257', '31', '314', '369', '383', '412', '450', '462', '577', '610', '800', '901', '909', '961']
EXCLUDED_CASES = ['15', '178', '300', '735', '77', '78', '856', '857', '912', '931', '970', '976', '990']
EXPECTED_MANIFEST_SHA256 = 'a8caa6ad39310c2871e53bc4b9f80d4b76c1b6f850b1c474c0abae8db1181283'

In [ ]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
with log_stage("Dataset discovery and extraction"):
    SOURCE_ROOT = discover_dataset(DATA_ROOT, INPUT_ROOT, EXTRACT_ROOT)
log("Dataset found:", SOURCE_ROOT)
metadata_path = SOURCE_ROOT / "bus_data.csv"
metadata_sha = sha_file(metadata_path)
variants = [name for name, spec in VERIFIED_DATASET_VARIANTS.items()
            if spec["metadata_sha256"] == metadata_sha]
if len(variants) != 1:
    raise ValueError("Dataset metadata differs from the verified BUS-BRA copies")
metadata = read_csv(metadata_path)
by_id = {r["ID"]: r for r in metadata}
if len(metadata) != 1875 or len(by_id) != len(metadata):
    raise ValueError("Unexpected metadata size or duplicate image IDs")
image_dirs = [p for p in SOURCE_ROOT.iterdir() if p.is_dir() and p.name.lower() == "images"]
if len(image_dirs) != 1:
    raise ValueError("Expected one Images directory beside bus_data.csv")
image_paths = defaultdict(list)
for path in image_dirs[0].rglob("*"):
    if path.is_file() and path.suffix.lower() in {".png", ".jpg", ".jpeg", ".bmp"}:
        image_paths[path.stem].append(path)
SAMPLE = []
with log_stage("Load the frozen 400-image final sample"):
    for image_id in FROZEN_IMAGE_IDS:
        row = by_id[image_id]
        if row["Case"] in set(PILOT_CASES) | set(EXCLUDED_CASES):
            raise ValueError("Final sample overlaps a pilot or excluded Case")
        paths = image_paths[image_id]
        if len(paths) != 1:
            raise ValueError(f"Missing or ambiguous image: {image_id}")
        path = paths[0]
        SAMPLE.append({"dataset": "bus_bra", "sample_id": image_id, "image_id": image_id,
                       "image_path": path.relative_to(SOURCE_ROOT).as_posix(),
                       "group_id": row["Case"], "split": "test", "task_type": "classification",
                       "question": BUS_BRA_PROMPT, "options_json": "{}",
                       "reference_answer": row["Pathology"].strip().lower(), "category": "",
                       "image_sha256": sha_file(path)})
    if (len(SAMPLE) != FINAL_SIZE or len({r["sample_id"] for r in SAMPLE}) != FINAL_SIZE
            or len({r["group_id"] for r in SAMPLE}) != FINAL_SIZE
            or Counter(r["reference_answer"] for r in SAMPLE) != {"benign": 200, "malignant": 200}):
        raise ValueError("Final sample must have 400 unique images/Cases and 200 images per class")
    manifest_path = OUTPUT_ROOT / "final_manifest.csv"
    write_csv(manifest_path, SAMPLE, MANIFEST_FIELDS)
    MANIFEST_SHA = sha_file(manifest_path)
    if MANIFEST_SHA != EXPECTED_MANIFEST_SHA256:
        raise ValueError("Frozen sample mismatch: image content, labels, Cases, paths or prompt changed")
DATA_IDENTITY = {"metadata_sha256": metadata_sha, "dataset_variant": variants[0],
                 "selected_manifest_sha256": MANIFEST_SHA}
write_json(OUTPUT_ROOT / "sample_metadata.json", {
    **DATA_IDENTITY, "purpose": "blank_image_control", "visual_inputs": BLANK_IMAGE_SPECS,
    "original_images_used_only_for_manifest_check": True, "seed": SEED, "images": FINAL_SIZE,
    "class_counts": dict(Counter(r["reference_answer"] for r in SAMPLE)),
    "grouping_field": "Case", "excluded_pilot_cases": PILOT_CASES,
    "excluded_audit_cases": EXCLUDED_CASES,
    "selection": "200 eligible Cases/class using random.Random(42); one random image/Case; sorted by image ID",
})
log("Final sample:", FINAL_SIZE, "images / 400 source Cases; 200 benign + 200 malignant")
log("Excluded pilot Cases:", len(PILOT_CASES), "| previously flagged Cases:", len(EXCLUDED_CASES))
log("Dataset variant:", variants[0], "| metadata SHA256:", metadata_sha)
log("Final manifest SHA256:", MANIFEST_SHA)
log("Fixed prompt:", repr(BUS_BRA_PROMPT))
log("CONTROL VISUAL INPUTS:", BLANK_IMAGE_SPECS)
log("Original image paths/hashes are checked for pairing, never opened during inference.")


## 4. Parse the model's exact final label

Retains the same strict output parser as the original experiment: only an unambiguous final `benign` or `malignant` counts as parsed. All raw model outputs are preserved.


In [ ]:
def decode_answer_tokens(tokenizer, tokens, eos_ids):
    """Remove only configured terminal EOS IDs from the parsing copy; retain raw text."""
    eos_ids = eos_ids if isinstance(eos_ids, (list, tuple)) else [eos_ids]
    eos_set = {token for token in eos_ids if token is not None}
    answer_tokens = list(tokens)
    removed = []
    while answer_tokens and answer_tokens[-1] in eos_set:
        removed.insert(0, answer_tokens.pop())
    kwargs = dict(clean_up_tokenization_spaces=False)
    return {
        "raw_output": tokenizer.decode(tokens, skip_special_tokens=True, **kwargs),
        "raw_output_with_special_tokens": tokenizer.decode(tokens, skip_special_tokens=False, **kwargs),
        "answer_text": tokenizer.decode(answer_tokens, skip_special_tokens=True, **kwargs),
        "terminal_eos_token_ids_removed": removed,
    }

def parse_answer(text):
    """Strict, deterministic final-answer parser; no label search inside reasoning."""
    response = text.strip()
    if not response:
        return None, "empty"
    # Exactly one balanced reasoning block may precede a final answer.
    if response.startswith("<think>"):
        if response.count("<think>") != 1 or response.count("</think>") != 1:
            return None, "incomplete_or_multiple_think_blocks"
        response = response.split("</think>", 1)[1].strip()
    if "<think>" in response or "</think>" in response:
        return None, "misplaced_think_tag"
    if "<answer>" in response or "</answer>" in response:
        match = re.fullmatch(r"<answer>\s*(benign|malignant)\s*</answer>", response, re.IGNORECASE)
        if match:
            return match.group(1).lower(), "answer_tag"
        return None, "malformed_or_ambiguous_answer_tag"
    match = re.fullmatch(r"(?:final answer:\s*)?(benign|malignant)", response, re.IGNORECASE)
    if match:
        return match.group(1).lower(), "direct_or_final_marker"
    return None, "no_unambiguous_final_answer"

## 5. Model loading and deterministic generation

Uses the same pinned checkpoints, **BF16**, the same prompt, and `do_sample=False` greedy decoding as the original notebook. Only the synthetic image's pixel color changes across the two conditions.


In [ ]:
with log_stage("ML library imports and GPU setup"):
    import importlib.metadata as im
    import torch
    from huggingface_hub import HfApi
    from transformers import (AutoProcessor, Qwen2VLForConditionalGeneration,
                              Gemma3ForConditionalGeneration,
                              GenerationConfig, LogitsProcessor, LogitsProcessorList)

    if not torch.cuda.is_available():
        raise RuntimeError("Enable a Kaggle GPU accelerator, then restart and Run All.")
    random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    # Greedy generation is reproducible under a fixed setup, not guaranteed bit-identical across hardware.
    HF_TOKEN = None
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        # Public Qwen downloads may still work. Gemma access failure is recorded per job.
        log("HF_TOKEN is unavailable. Enable that Kaggle secret for the gated Gemma models.")

    BF16_CAPABILITIES = []
    for gpu_index in range(torch.cuda.device_count()):
        with torch.cuda.device(gpu_index):
            capability = {
                "gpu": gpu_index,
                "native_supported": torch.cuda.is_bf16_supported(including_emulation=False),
                "including_emulation_supported": torch.cuda.is_bf16_supported(including_emulation=True),
                "probe_scope": "PyTorch capability/tensor-allocation check; not full operator validation",
            }
            capability["execution_expectation"] = (
                "native_bf16_available" if capability["native_supported"]
                else "non_native_bf16_requested")
            BF16_CAPABILITIES.append(capability)
            log("BF16 capability:", capability)
    log("BF16 is requested for every job. Using the BF16 path validated in v7; no precision fallback.")

    ENVIRONMENT = {
        "python": platform.python_version(), "platform": platform.platform(),
        "torch": torch.__version__, "cuda_runtime": torch.version.cuda,
        "preserved_runtime_packages": dict(PRESERVED_RUNTIME_VERSIONS),
        "packages": {d.metadata["Name"]: d.version for d in im.distributions() if d.metadata["Name"]},
        "gpus": [{"index": i, "name": torch.cuda.get_device_name(i),
                  "total_bytes": torch.cuda.get_device_properties(i).total_memory,
                  "compute_capability": list(torch.cuda.get_device_capability(i))}
                 for i in range(torch.cuda.device_count())],
        "tf32_enabled": False, "bf16_capabilities": BF16_CAPABILITIES,
    }
    try:
        ENVIRONMENT["nvidia_smi"] = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
            capture_output=True, text=True, check=False, timeout=20).stdout.strip()
    except (OSError, subprocess.TimeoutExpired):
        ENVIRONMENT["nvidia_smi"] = "unavailable"
    # Full package inventory is saved; only inference-relevant versions identify resume compatibility.
    RUNTIME_IDENTITY = {"python": ENVIRONMENT["python"], "torch": ENVIRONMENT["torch"],
                        "cuda_runtime": ENVIRONMENT["cuda_runtime"], "gpus": ENVIRONMENT["gpus"],
                        "packages": {p: im.version(p) for p in [*PACKAGE_PINS, *PRESERVED_RUNTIME_VERSIONS, "numpy"]},
                        "bf16_capabilities": BF16_CAPABILITIES}
    write_json(OUTPUT_ROOT / "environment.json", ENVIRONMENT)
    log("GPUs:", [(g["index"], g["name"]) for g in ENVIRONMENT["gpus"]])
    log("No CPU/disk offloading or automatic precision fallback will be used.")

def sanitized_error(exc):
    message = str(exc)
    if HF_TOKEN:
        message = message.replace(HF_TOKEN, "[REDACTED]")
    message = re.sub(r"hf_[A-Za-z0-9]+", "[REDACTED]", message)
    return {"type": type(exc).__name__, "message": message[:4000]}

def cleanup_cuda():
    gc.collect()
    torch.cuda.empty_cache()

def sync_cuda():
    for i in range(torch.cuda.device_count()):
        torch.cuda.synchronize(i)

def reset_peaks():
    sync_cuda()
    for i in range(torch.cuda.device_count()):
        torch.cuda.reset_peak_memory_stats(i)

def memory_snapshot():
    return [{"gpu": i, "peak_allocated_bytes": torch.cuda.max_memory_allocated(i),
             "peak_reserved_bytes": torch.cuda.max_memory_reserved(i)}
            for i in range(torch.cuda.device_count())]

class NumericsProbe(LogitsProcessor):
    """Observe generation scores without modifying decoding or retaining tensors."""
    def __init__(self):
        self.steps = 0
        self.nonfinite_steps = 0
        self.nan_or_posinf_steps = 0
        self.all_masked_steps = 0

    def __call__(self, input_ids, scores):
        self.steps += 1
        self.nonfinite_steps += int((~torch.isfinite(scores)).any().item())
        self.nan_or_posinf_steps += int((torch.isnan(scores) | torch.isposinf(scores)).any().item())
        self.all_masked_steps += int(torch.isneginf(scores).all(dim=-1).any().item())
        return scores

def resolve_revision(key):
    path = OUTPUT_ROOT / "model_revisions.json"
    saved = json.loads(path.read_text()) if path.exists() else {}
    requested = MODEL_REVISIONS[key]
    if key in saved:
        if saved[key].get("model_id") != MODEL_IDS[key] or not re.fullmatch(r"[0-9a-f]{40}", saved[key].get("revision", "")):
            raise ValueError("Invalid saved checkpoint identity")
        if requested is not None and requested != saved[key]["revision"]:
            raise ValueError("Requested revision differs from frozen run; use a fresh OUTPUT_ROOT")
        return saved[key]["revision"]
    with log_stage(f"{key}: resolving checkpoint revision"):
        revision = HfApi(token=HF_TOKEN or False).model_info(MODEL_IDS[key], revision=requested).sha
    if not re.fullmatch(r"[0-9a-f]{40}", revision or ""):
        raise ValueError("Could not resolve exact model commit")
    if revision != requested:
        raise ValueError("Resolved checkpoint differs from the frozen revision")
    saved[key] = {"model_id": MODEL_IDS[key], "revision": revision, "resolved_at": utc_now()}
    write_json(path, saved)
    return revision

def load_model(key, precision, revision):
    if precision != "bf16":
        raise ValueError("Final inference only permits BF16 inference")
    dtype = torch.bfloat16
    # Use slow/native processor consistently to avoid silently changing image backends.
    cache = MODEL_CACHE_ROOT / key
    cache.mkdir(parents=True, exist_ok=True)
    (cache / ".bus_bra_owned_cache").touch()
    with log_stage(f"{key}/{precision}: processor download and load"):
        processor = AutoProcessor.from_pretrained(MODEL_IDS[key], revision=revision,
                                                  token=HF_TOKEN or False, use_fast=False, cache_dir=cache)
    budget = {i: int(torch.cuda.mem_get_info(i)[0] * GPU_MEMORY_FRACTION)
              for i in range(torch.cuda.device_count())}
    kwargs = dict(revision=revision, token=HF_TOKEN or False, torch_dtype=dtype,
                  attn_implementation="sdpa", device_map="auto", max_memory={**budget, "cpu": 0},
                  low_cpu_mem_usage=True, cache_dir=cache)
    cls = Qwen2VLForConditionalGeneration if key in {"qwen2_vl", "medvlm_r1"} else Gemma3ForConditionalGeneration
    with log_stage(f"{key}/{precision}: model download and load"):
        model = cls.from_pretrained(MODEL_IDS[key], **kwargs).eval()
    device_map = {k: str(v) for k, v in model.hf_device_map.items()}
    if any(v in {"cpu", "disk"} for v in device_map.values()):
        raise RuntimeError("Model was offloaded to CPU/disk; final inference requires GPU-only inference")
    details = {"device_map": device_map, "gpu_weight_budgets_bytes": budget,
               "parameter_elements_by_dtype": dict(Counter()),
               "quantization": model.config.to_dict().get("quantization_config"),
               "image_processor": processor.image_processor.to_dict(),
               "processor_class": type(processor).__name__,
               "chat_template": processor.chat_template,
               "special_tokens_map": processor.tokenizer.special_tokens_map,
               "generation_defaults": model.generation_config.to_dict()}
    dtypes = Counter()
    for parameter in model.parameters():
        dtypes[str(parameter.dtype)] += parameter.numel()
    details["parameter_elements_by_dtype"] = dict(dtypes)
    details["requested_dtype"] = str(dtype)
    details["bf16_capabilities"] = BF16_CAPABILITIES
    if dtypes.get("torch.bfloat16", 0) == 0:
        raise RuntimeError("BF16 was requested but no BF16 model parameters were loaded")
    log(key, "loaded parameter dtypes:", dict(dtypes))
    quant_layers = [{"name": name, "compute_dtype": str(layer.compute_dtype)}
                    for name, layer in model.named_modules() if hasattr(layer, "compute_dtype")]
    details["quantized_compute_layers"] = quant_layers
    return model, processor, details, dtype

def prepare_checked_generation(model, tokenizer):
    # Explicit kwargs outrank checkpoint defaults in Transformers 4.57.1.
    # Disabling model-default substitution also protects values equal to global defaults.
    controls = {"max_new_tokens": MAX_NEW_TOKENS, "do_sample": False,
                "num_beams": 1, "num_return_sequences": 1, "use_cache": True,
                "temperature": 1.0, "top_k": 50, "top_p": 1.0}
    eos = model.generation_config.eos_token_id
    if eos is None:
        eos = tokenizer.eos_token_id
    pad = tokenizer.pad_token_id
    if pad is None:
        pad = eos[0] if isinstance(eos, (list, tuple)) else eos
    requested = GenerationConfig(**controls, eos_token_id=eos, pad_token_id=pad,
                                 bos_token_id=tokenizer.bos_token_id)
    # Inspect the installed library's actual resolution path, not just the caller object.
    resolved, leftovers = model._prepare_generation_config(
        requested, use_model_defaults=False, **controls)
    mismatches = {k: {"expected": v, "resolved": getattr(resolved, k, None)}
                  for k, v in controls.items() if getattr(resolved, k, None) != v}
    if mismatches or leftovers:
        raise RuntimeError(f"Generation settings were not enforced: {mismatches}; unused={leftovers}")
    if resolved.eos_token_id != eos or resolved.pad_token_id != pad:
        raise RuntimeError("Generation resolution changed explicit EOS/pad token IDs")
    if resolved.get_generation_mode().value != "greedy_search":
        raise RuntimeError("Resolved generation mode is not greedy_search")
    return resolved, controls


def infer_one(model, processor, dtype, control_color):
    # No image path, sample ID, metadata, or ground-truth label enters inference.
    # The five repetitions for each condition receive identical fresh image pixels.
    reset_peaks()
    started = time.perf_counter()
    if control_color not in CONTROL_COLORS:
        raise ValueError(f"Unknown control condition: {control_color}")
    image = Image.new("RGB", BLANK_IMAGE_SIZE, CONTROL_COLORS[control_color])
    spec = BLANK_IMAGE_SPECS[control_color]
    assert sha_bytes(image.tobytes()) == spec["pixel_sha256"], "Unexpected control image pixels"
    messages = [{"role": "user", "content": [{"type": "image"},
                 {"type": "text", "text": BUS_BRA_PROMPT}]}]
    rendered = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[rendered], images=[image], return_tensors="pt", padding=True)
    input_details = {k: list(v.shape) for k, v in inputs.items() if hasattr(v, "shape")}
    device = model.get_input_embeddings().weight.device
    inputs = {k: v.to(device=device, dtype=dtype) if v.is_floating_point() else v.to(device)
              for k, v in inputs.items()}
    input_len = inputs["input_ids"].shape[1]
    generation, controls = prepare_checked_generation(model, processor.tokenizer)
    eos = generation.eos_token_id
    if not getattr(model, "_bus_bra_generation_logged", False):
        log("Resolved generation:", {"use_model_defaults": False,
            "generation_mode": generation.get_generation_mode().value, **controls,
            "eos_token_id": eos, "pad_token_id": generation.pad_token_id,
            "bos_token_id": generation.bos_token_id})
        model._bus_bra_generation_logged = True
    probe = NumericsProbe()
    with torch.inference_mode():
        output = model.generate(**inputs, generation_config=generation,
                                use_model_defaults=False, **controls,
                                logits_processor=LogitsProcessorList([probe]))
    sync_cuda()
    latency_ms = (time.perf_counter() - started) * 1000
    tokens = output[0, input_len:].detach().cpu().tolist()
    eos_ids = eos if isinstance(eos, list) else [eos]
    decoded = decode_answer_tokens(processor.tokenizer, tokens, eos_ids)
    stop = "eos" if tokens and tokens[-1] in eos_ids else "max_new_tokens" if len(tokens) >= MAX_NEW_TOKENS else "other"
    answer, reason = parse_answer(decoded["answer_text"])
    bad_numerics = probe.nan_or_posinf_steps > 0 or probe.all_masked_steps > 0
    if bad_numerics:
        answer, reason = None, "invalid_generation_scores"
    counts = Counter(tokens)
    return {**decoded, "control_color": control_color,
            "visual_input_mode": spec["mode"],
            "visual_input_pixel_sha256": spec["pixel_sha256"],
            "parser_version": PARSER_VERSION,
            "parsed_answer": answer, "parse_status": "parsed" if answer is not None else "unparsed",
            "parse_reason": reason, "generated_tokens": len(tokens), "generated_token_ids": tokens,
            "stop_reason": stop, "hit_token_limit": stop == "max_new_tokens",
            "possible_answer_truncation": stop == "max_new_tokens" and answer is None,
            "latency_ms": latency_ms, "peak_memory": memory_snapshot(),
            "input_shapes": input_details, "input_dtypes": {k: str(v.dtype) for k, v in inputs.items()},
            "rendered_prompt_sha256": sha_bytes(rendered.encode()),
            "score_diagnostics": vars(probe), "invalid_numerics": bad_numerics,
            "repetition_candidate": len(tokens) >= 8 and max(counts.values(), default=0) / len(tokens) >= 0.5,
            "generation_config": generation.to_dict(), "use_model_defaults": False,
            "generation_mode": generation.get_generation_mode().value}


## 6. Run 5 white + 5 black control inferences per model

The results are saved **after every inference**, separately from the earlier experiment. If the five repetitions disagree on the raw or parsed answer, the notebook reports instability and **does not fabricate 400 identical predictions** for that model/color.

When stable, one verified prediction is copied to the **same 400 sample IDs and ground truths** as the original run. Those control CSV rows are derived for paired evaluation; the inference count remains five per condition.


In [ ]:
SAMPLE_COLUMNS = ["idx", "question_id", "group"]
RESULT_COLUMNS = ["idx", "question_id", "group", "anatomy", "question",
                  "true_answer", "predicted_answer", "raw_output"]
SCORE_COLUMNS = ["group", "correct", "n", "accuracy"]
EXPORT_GROUP = "lesion_classification"
TRIAL_COLUMNS = ["model", "model_revision", "color", "trial", "parsed_answer",
                 "parse_status", "parse_reason", "raw_output", "generated_tokens",
                 "latency_ms", "visual_input_pixel_sha256", "completed_at", "status"]
SUMMARY_COLUMNS = ["model", "color", "completed_trials", "expected_trials", "stable",
                   "parsed_prediction", "raw_output", "derived_rows", "derived_accuracy",
                   "unique_raw_outputs", "unique_parsed_predictions", "model_revision", "status"]

# The *same* 400 frozen image IDs / references are used to evaluate constant baselines.
_SOURCE_INDEX = {row["ID"]: i for i, row in enumerate(read_csv(SOURCE_ROOT / "bus_data.csv"))}
BASE_SAMPLE_ROWS = [
    {"idx": _SOURCE_INDEX[s["sample_id"]], "question_id": s["sample_id"], "group": EXPORT_GROUP}
    for s in SAMPLE
]
BASE_BY_ID = {row["question_id"]: row for row in BASE_SAMPLE_ROWS}
assert len(BASE_SAMPLE_ROWS) == 400 and len(BASE_BY_ID) == 400

TRIAL_RECORDS = []
SUMMARY_RECORDS = []
CSV_ROOT.mkdir(parents=True, exist_ok=True)
for control_color in CONTROL_COLORS:
    color_root = CSV_ROOT / control_color
    color_root.mkdir(parents=True, exist_ok=True)
    write_csv(color_root / "vqa_sample.csv", BASE_SAMPLE_ROWS, SAMPLE_COLUMNS)


def save_trial(trial):
    # Incremental saving preserves work even when the next inference fails.
    TRIAL_RECORDS.append(trial)
    write_csv(CSV_ROOT / "control_trials.csv", TRIAL_RECORDS, TRIAL_COLUMNS)


def export_control_model(color, key, trials, model_revision):
    """Expand a reproducible constant response to 400 *derived*, not inferred, rows."""
    tag = EXPORT_TAGS[key]
    out_dir = CSV_ROOT / color
    valid = [t for t in trials if t.get("status") == "ok"]
    raw_set = {t["raw_output"] for t in valid}
    prediction_set = {(t["parsed_answer"], t["parse_status"], t["parse_reason"])
                      for t in valid}
    stable = (len(trials) == TRIALS_PER_COLOR and len(valid) == TRIALS_PER_COLOR
              and len(raw_set) == 1 and len(prediction_set) == 1)
    label = (valid[0]["parsed_answer"] or "unparsed") if stable else ""
    rows = []
    if stable:
        for sample in SAMPLE:
            rows.append({**BASE_BY_ID[sample["sample_id"]],
                         "anatomy": "breast", "question": sample["question"],
                         "true_answer": sample["reference_answer"],
                         "predicted_answer": label,
                         "raw_output": valid[0]["raw_output"]})
    correct = sum(r["true_answer"] == r["predicted_answer"] for r in rows)
    accuracy = correct / len(rows) if rows else None
    # Empty CSV with correct header is intentional if the stability check fails.
    write_csv(out_dir / f"vqa_results_{tag}.csv", rows, RESULT_COLUMNS)
    write_csv(out_dir / f"vqa_scores_{tag}.csv",
              [{"group": EXPORT_GROUP, "correct": correct, "n": len(rows), "accuracy": accuracy}],
              SCORE_COLUMNS)
    write_json(out_dir / f"derived_control_provenance_{tag}.json", {
        "model": key, "model_id": MODEL_IDS[key], "model_revision": model_revision,
        "condition": color, "condition_spec": BLANK_IMAGE_SPECS[color],
        "prompt": BUS_BRA_PROMPT, "generation": "greedy do_sample=False",
        "actual_inferences": len(valid), "requested_repetitions": TRIALS_PER_COLOR,
        "derived_prediction_rows": len(rows), "derived_not_independently_inferred": True,
        "stable_raw_and_parsed_output": stable, "raw_outputs_observed": len(raw_set),
        "unique_parsed_outputs": len(prediction_set), "manifest_sha256": MANIFEST_SHA,
        "code_version": CODE_VERSION,
    })
    summary = {"model": key, "color": color,
               "completed_trials": len(valid), "expected_trials": TRIALS_PER_COLOR,
               "stable": stable, "parsed_prediction": label if stable else "unstable",
               "raw_output": valid[0]["raw_output"] if stable else "",
               "derived_rows": len(rows), "derived_accuracy": accuracy,
               "unique_raw_outputs": len(raw_set), "unique_parsed_predictions": len(prediction_set),
               "model_revision": model_revision,
               "status": "ready_for_paired_comparison" if stable else "unstable_or_incomplete"}
    return summary


def run_dual_control_job(key, precision):
    model = processor = None
    revisions = None
    current_color = None
    fatal_cuda = False
    result = []
    run_dir = OUTPUT_ROOT / "dual_control_runs" / key
    run_dir.mkdir(parents=True, exist_ok=True)
    trials_for_key = {color: [] for color in CONTROL_COLORS}
    try:
        cleanup_cuda()
        revisions = resolve_revision(key)
        with log_stage(f"{key}/{precision}: load model and processor"):
            model, processor, model_details, dtype = load_model(key, precision, revisions)
        write_json(run_dir / "model_details.json", model_details)
        for color in CONTROL_COLORS:
            current_color = color
            log(f"{key}: control={color}, RGB={CONTROL_COLORS[color]}")
            for trial_i in range(1, TRIALS_PER_COLOR + 1):
                try:
                    with log_stage(f"{key}: {color} {trial_i}/{TRIALS_PER_COLOR}"):
                        payload = infer_one(model, processor, dtype, control_color=color)
                    trial = {"model": key, "model_revision": revisions, "color": color,
                             "trial": trial_i, "parsed_answer": payload["parsed_answer"] or "",
                             "parse_status": payload["parse_status"],
                             "parse_reason": payload["parse_reason"],
                             "raw_output": payload["raw_output"],
                             "generated_tokens": payload["generated_tokens"],
                             "latency_ms": payload["latency_ms"],
                             "visual_input_pixel_sha256": payload["visual_input_pixel_sha256"],
                             "completed_at": utc_now(), "status": "ok"}
                    save_trial(trial)
                    trials_for_key[color].append(trial)
                    append_jsonl(run_dir / "generations.jsonl", {**trial, **payload})
                    log(key, color, trial_i, "=>", repr(payload["raw_output"]),
                        "/ parsed:", payload["parsed_answer"])
                except Exception as exc:
                    log("ERROR", key, color, trial_i, sanitized_error(exc))
                    append_jsonl(run_dir / "errors.jsonl", {"model": key, "color": color,
                        "trial": trial_i, "error": sanitized_error(exc), "timestamp": utc_now()})
                    if any(s in str(exc).lower() for s in ("device-side assert", "illegal memory access", "out of memory")):
                        fatal_cuda = True
                    break
            result.append(export_control_model(color, key, trials_for_key[color], revisions))
            write_csv(CSV_ROOT / "control_summary.csv", [*SUMMARY_RECORDS, *result], SUMMARY_COLUMNS)
            if fatal_cuda:
                break
    finally:
        model = processor = None
        if not fatal_cuda:
            cleanup_cuda()
    return result, fatal_cuda


## 7. Execute both model jobs

**Expected VLM calls: 2 models × 2 colors × 5 repetitions = 20**, not 800. Both original-image CSVs remain untouched. The original 400 rows are used only when deriving paired-control predictions.


In [ ]:
SESSION_RESULTS = []
for job_index, (key, precision) in enumerate(JOBS, start=1):
    log(f"--- Model {job_index}/{len(JOBS)}: {key} ({precision}) ---")
    try:
        summaries, fatal_cuda = run_dual_control_job(key, precision)
    except Exception as exc:
        log("MODEL JOB FAILED", key, sanitized_error(exc))
        append_jsonl(OUTPUT_ROOT / "job_errors.jsonl", {"model": key, "error": sanitized_error(exc)})
        summaries, fatal_cuda = [], any(s in str(exc).lower() for s in ("device-side assert", "illegal memory access"))
    SUMMARY_RECORDS.extend(summaries)
    SESSION_RESULTS.append({"model": key, "summaries": summaries, "fatal_cuda": fatal_cuda})
    write_csv(CSV_ROOT / "control_summary.csv", SUMMARY_RECORDS, SUMMARY_COLUMNS)
    if CLEAR_MODEL_CACHE_AFTER_LAST_JOB:
        cache = MODEL_CACHE_ROOT / key
        if cache.is_dir() and not cache.is_symlink() and (cache / ".bus_bra_owned_cache").is_file():
            log("Removing owned model cache", key)
            shutil.rmtree(cache)
    if fatal_cuda:
        log("Stopping after fatal CUDA error. Partial trial logs are retained.")
        break
log("Completed control inferences:", len(TRIAL_RECORDS), "of", len(JOBS) * len(CONTROL_COLORS) * TRIALS_PER_COLOR)


## 8. Inspect and download CSVs

`control_trials.csv` contains **actual model responses** (expected 20 rows total). `control_summary.csv` documents stability by model/color. Under each `white/` and `black/` subdirectory, `vqa_results_*.csv` and `vqa_scores_*.csv` are **derived matched 400-row control baselines** when all five repetitions agree.

To compare with previously saved real-image results, run your existing real-vs-blank paired analysis notebook twice, pointing its blank-results folder first to `white/` and then to `black/`.


In [ ]:
import pandas as pd
from IPython.display import FileLink, display

summary_frame = pd.DataFrame(SUMMARY_RECORDS)
if not summary_frame.empty:
    display(summary_frame)
else:
    log("No completed model/color summaries yet; inspect the job error logs.")

all_paths = [CSV_ROOT / "control_trials.csv", CSV_ROOT / "control_summary.csv"]
for color in CONTROL_COLORS:
    all_paths.append(CSV_ROOT / color / "vqa_sample.csv")
    for key, _ in JOBS:
        tag = EXPORT_TAGS[key]
        all_paths.extend([CSV_ROOT / color / f"vqa_results_{tag}.csv",
                          CSV_ROOT / color / f"vqa_scores_{tag}.csv",
                          CSV_ROOT / color / f"derived_control_provenance_{tag}.json"])
for path in all_paths:
    if not path.exists():
        continue
    try:
        display(FileLink(str(path.relative_to(Path.cwd()))))
    except ValueError:
        log("Download from Kaggle Output files:", path)

expected_summaries = len(JOBS) * len(CONTROL_COLORS)
ready = (len(SUMMARY_RECORDS) == expected_summaries
         and all(s["derived_rows"] == FINAL_SIZE and s["stable"] for s in SUMMARY_RECORDS))
if not ready:
    raise RuntimeError("One or more controls were unstable/incomplete. Do not treat missing derived CSVs as final predictions. Inspect control_trials.csv and errors.")
log("COMPLETE: 20 actual control inferences; four separate, derived 400-row CSVs ready for paired analysis.")
